# FINAL INSIGHTS & BUSINESS RECOMMENDATIONS

This notebook synthesizes findings from all previous analyses and provides actionable business recommendations based on the A/B test results.

**Summary of Analysis Pipeline:**

| Notebook | Topic |
|---|---|
| 01 | Data Understanding |
| 02 | Exploratory Data Analysis |
| 03 | Statistical Inference (A/B Test) |
| 04 | Resampling Methods (Bootstrap) |
| 05 | Power Analysis |
| 06 | Regression Analysis |
| 07 | Classification Model |
| **08** | **Final Insights** |


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')

In [ ]:
df = pd.read_csv('../data/processed/cleaned_marketing.csv')
df['date'] = pd.to_datetime(df['date'])

df_clean = df.dropna().copy()
control = df_clean[df_clean['group'] == 'control']
test = df_clean[df_clean['group'] == 'test']

print(f'Total records (clean): {len(df_clean)}')
print(f'Control group:         {len(control)} days')
print(f'Test group:            {len(test)} days')

## 1. Campaign Performance Dashboard


In [ ]:
metrics = ['spend_usd', 'impressions', 'reach', 'website_clicks',
           'searches', 'view_content', 'add_to_cart', 'purchase']

summary = pd.DataFrame({
    'Control Mean': control[metrics].mean(),
    'Test Mean': test[metrics].mean(),
    'Control Std': control[metrics].std(),
    'Test Std': test[metrics].std()
})
summary['% Difference'] = ((summary['Test Mean'] - summary['Control Mean']) / summary['Control Mean'] * 100).round(2)
summary = summary.round(2)
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

labels = ['Control', 'Test']
colors = ['#4C72B0', '#DD8452']

for i, metric in enumerate(metrics):
    means = [control[metric].mean(), test[metric].mean()]
    stds = [control[metric].std(), test[metric].std()]

    axes[i].bar(labels, means, color=colors, edgecolor='black', alpha=0.85, width=0.5)
    axes[i].errorbar(labels, means, yerr=stds, fmt='none', color='black', capsize=5)

    pct_diff = (means[1] - means[0]) / means[0] * 100
    direction = '▲' if pct_diff > 0 else '▼'
    axes[i].set_title(f'{metric.replace("_", " ").title()}\n{direction} {abs(pct_diff):.1f}%', fontsize=10)
    axes[i].set_ylabel('Mean Value')

plt.suptitle('Campaign Performance: Control vs Test (Mean ± Std)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 2. Statistical Test Results Summary


In [ ]:
results = []
for metric in metrics:
    c_vals = control[metric].dropna()
    t_vals = test[metric].dropna()

    t_stat, p_ttest = ttest_ind(c_vals, t_vals, equal_var=False)
    u_stat, p_mwu = mannwhitneyu(c_vals, t_vals, alternative='two-sided')

    mean_diff = t_vals.mean() - c_vals.mean()
    pooled_std = np.sqrt((c_vals.std()**2 + t_vals.std()**2) / 2)
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0

    results.append({
        'Metric': metric,
        'Control Mean': round(c_vals.mean(), 2),
        'Test Mean': round(t_vals.mean(), 2),
        'Mean Diff': round(mean_diff, 2),
        'T-test p-value': round(p_ttest, 4),
        'MWU p-value': round(p_mwu, 4),
        "Cohen's d": round(cohens_d, 4),
        'Significant (p<0.05)': '✓' if p_ttest < 0.05 else '✗'
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 3. Conversion Funnel Analysis


In [ ]:
funnel_stages = ['impressions', 'reach', 'website_clicks', 'searches', 'view_content', 'add_to_cart', 'purchase']

control_funnel = control[funnel_stages].mean()
test_funnel = test[funnel_stages].mean()

# Normalize to impressions = 100%
control_pct = (control_funnel / control_funnel['impressions'] * 100).round(2)
test_pct = (test_funnel / test_funnel['impressions'] * 100).round(2)

funnel_df = pd.DataFrame({
    'Stage': funnel_stages,
    'Control %': control_pct.values,
    'Test %': test_pct.values
})
print('Funnel Conversion (relative to impressions):')
print(funnel_df.to_string(index=False))

In [ ]:
x = np.arange(len(funnel_stages))
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, control_funnel.values, width, label='Control', color='#4C72B0', alpha=0.85)
plt.bar(x + width/2, test_funnel.values, width, label='Test', color='#DD8452', alpha=0.85)

plt.xticks(x, [s.replace('_', ' ').title() for s in funnel_stages], rotation=30, ha='right')
plt.ylabel('Average Count per Day')
plt.title('Marketing Funnel: Average Daily Performance')
plt.legend()
plt.yscale('log')
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Step-to-step conversion rates
def step_conversion(series, stages):
    rates = {}
    for i in range(1, len(stages)):
        prev = series[stages[i-1]]
        curr = series[stages[i]]
        rates[f'{stages[i-1]} → {stages[i]}'] = round(curr / prev * 100, 2) if prev > 0 else 0
    return rates

ctrl_conv = step_conversion(control_funnel, funnel_stages)
test_conv = step_conversion(test_funnel, funnel_stages)

conv_df = pd.DataFrame({
    'Step': list(ctrl_conv.keys()),
    'Control Rate (%)': list(ctrl_conv.values()),
    'Test Rate (%)': list(test_conv.values())
})
conv_df['Diff (pp)'] = (conv_df['Test Rate (%)'] - conv_df['Control Rate (%)']).round(2)
print('Step-by-Step Conversion Rates:')
print(conv_df.to_string(index=False))

## 4. Spend Efficiency (ROI Proxy)


In [ ]:
# Cost per acquisition (CPA) = spend / purchases
control_cpa = (control['spend_usd'] / control['purchase']).mean()
test_cpa = (test['spend_usd'] / test['purchase']).mean()

# Cost per click (CPC)
control_cpc = (control['spend_usd'] / control['website_clicks']).mean()
test_cpc = (test['spend_usd'] / test['website_clicks']).mean()

# Click-through rate: clicks / impressions
control_ctr = (control['website_clicks'] / control['impressions'] * 100).mean()
test_ctr = (test['website_clicks'] / test['impressions'] * 100).mean()

# Purchase rate: purchases / clicks
control_pr = (control['purchase'] / control['website_clicks'] * 100).mean()
test_pr = (test['purchase'] / test['website_clicks'] * 100).mean()

efficiency = pd.DataFrame({
    'Metric': ['CPA ($/purchase)', 'CPC ($/click)', 'CTR (%)', 'Purchase Rate (%)'],
    'Control': [round(control_cpa, 2), round(control_cpc, 4), round(control_ctr, 4), round(control_pr, 4)],
    'Test': [round(test_cpa, 2), round(test_cpc, 4), round(test_ctr, 4), round(test_pr, 4)]
})
efficiency['% Change'] = ((efficiency['Test'] - efficiency['Control']) / efficiency['Control'] * 100).round(2)
efficiency['Better Campaign'] = efficiency.apply(
    lambda row: 'Test ✓' if (
        (row['Metric'] in ['CPA ($/purchase)', 'CPC ($/click)'] and row['% Change'] < 0) or
        (row['Metric'] not in ['CPA ($/purchase)', 'CPC ($/click)'] and row['% Change'] > 0)
    ) else 'Control ✓',
    axis=1
)

print('Spend Efficiency Metrics:')
print(efficiency.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CPA comparison
axes[0].bar(['Control', 'Test'], [control_cpa, test_cpa], color=['#4C72B0', '#DD8452'], edgecolor='black', alpha=0.85)
axes[0].set_title('Cost Per Acquisition (CPA)\nLower is Better')
axes[0].set_ylabel('USD per Purchase')
for i, v in enumerate([control_cpa, test_cpa]):
    axes[0].text(i, v * 0.95, f'${v:.2f}', ha='center', va='top', fontweight='bold', color='white')

# Purchase Rate comparison
axes[1].bar(['Control', 'Test'], [control_pr, test_pr], color=['#4C72B0', '#DD8452'], edgecolor='black', alpha=0.85)
axes[1].set_title('Purchase Rate (Purchases / Clicks)\nHigher is Better')
axes[1].set_ylabel('Rate (%)')
for i, v in enumerate([control_pr, test_pr]):
    axes[1].text(i, v * 0.95, f'{v:.2f}%', ha='center', va='top', fontweight='bold', color='white')

plt.tight_layout()
plt.show()

## 5. Time-Series Trend Analysis


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Purchase trend
axes[0].plot(control['date'], control['purchase'], 'o-', color='#4C72B0', label='Control', linewidth=2, markersize=5)
axes[0].plot(test['date'], test['purchase'], 's-', color='#DD8452', label='Test', linewidth=2, markersize=5)
axes[0].set_ylabel('Purchases')
axes[0].set_title('Daily Purchase Trend')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Spend trend
axes[1].plot(control['date'], control['spend_usd'], 'o-', color='#4C72B0', label='Control', linewidth=2, markersize=5)
axes[1].plot(test['date'], test['spend_usd'], 's-', color='#DD8452', label='Test', linewidth=2, markersize=5)
axes[1].set_ylabel('Spend (USD)')
axes[1].set_title('Daily Spend Trend')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# CTR trend
ctrl_ctr_daily = (control['website_clicks'] / control['impressions'] * 100)
test_ctr_daily = (test['website_clicks'] / test['impressions'] * 100)

axes[2].plot(control['date'], ctrl_ctr_daily, 'o-', color='#4C72B0', label='Control', linewidth=2, markersize=5)
axes[2].plot(test['date'], test_ctr_daily, 's-', color='#DD8452', label='Test', linewidth=2, markersize=5)
axes[2].set_ylabel('CTR (%)')
axes[2].set_title('Daily Click-Through Rate (CTR)')
axes[2].set_xlabel('Date')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 6. Final Summary Card


In [ ]:
# A/B Test result
t_stat, p_value = ttest_ind(control['purchase'], test['purchase'], equal_var=False)
significant = p_value < 0.05

mean_control = control['purchase'].mean()
mean_test = test['purchase'].mean()
pct_diff = (mean_test - mean_control) / mean_control * 100

pooled_std = np.sqrt((control['purchase'].std()**2 + test['purchase'].std()**2) / 2)
cohens_d = (mean_test - mean_control) / pooled_std

print('=' * 55)
print('       A/B TEST FINAL SUMMARY REPORT')
print('=' * 55)
print(f'Campaign Period:    {df_clean["date"].min().date()} → {df_clean["date"].max().date()}')
print(f'Control Group:      {len(control)} days | Mean Purchase = {mean_control:.2f}')
print(f'Test Group:         {len(test)} days | Mean Purchase = {mean_test:.2f}')
print('-' * 55)
print(f'Mean Difference:    {mean_test - mean_control:+.2f} ({pct_diff:+.2f}%)')
print(f'T-statistic:        {t_stat:.4f}')
print(f'P-value:            {p_value:.4f}')
print(f"Cohen's d:          {cohens_d:.4f} ({'Negligible' if abs(cohens_d) < 0.2 else 'Small' if abs(cohens_d) < 0.5 else 'Medium' if abs(cohens_d) < 0.8 else 'Large'})")
print('-' * 55)
print(f'Statistically Significant (p < 0.05): {"YES ✓" if significant else "NO ✗"}')
print(f'CPA Control:        ${control_cpa:.2f}/purchase')
print(f'CPA Test:           ${test_cpa:.2f}/purchase')
print(f'CTR Control:        {control_ctr:.4f}%')
print(f'CTR Test:           {test_ctr:.4f}%')
print('=' * 55)

## 7. Business Recommendations


In [ ]:
print("""
╔══════════════════════════════════════════════════════╗
║           BUSINESS RECOMMENDATIONS                  ║
╠══════════════════════════════════════════════════════╣
║                                                      ║
║  1. STATISTICAL SIGNIFICANCE                         ║
║     The A/B test found NO statistically significant  ║
║     difference in purchases between Control and      ║
║     Test campaigns (p ≥ 0.05).                       ║
║                                                      ║
║  2. PRACTICAL EFFECT                                 ║
║     Cohen's d is near 0, indicating negligible       ║
║     practical difference between campaigns.          ║
║                                                      ║
║  3. FUNNEL ANALYSIS                                  ║
║     Compare step-by-step conversion rates to         ║
║     identify where each campaign performs better.    ║
║     Focus optimization effort on the weakest stage.  ║
║                                                      ║
║  4. COST EFFICIENCY                                  ║
║     Choose the campaign with lower CPA and higher    ║
║     CTR for budget allocation decisions.             ║
║                                                      ║
║  5. SAMPLE SIZE                                      ║
║     The current sample (30 days per group) may be    ║
║     insufficient to detect small effects. Consider   ║
║     extending the experiment duration.               ║
║                                                      ║
║  6. NEXT STEPS                                       ║
║     a) Run experiment for 60+ days per group         ║
║     b) Segment analysis by day-of-week               ║
║     c) Consider multivariate testing (MVT)           ║
║     d) Track long-term customer retention            ║
╚══════════════════════════════════════════════════════╝
""")

## 8. Conclusion

### Key Findings

| Analysis | Finding |
|---|---|
| **A/B Test (t-test)** | No statistically significant difference in purchases (p ≥ 0.05) |
| **Bootstrap CI** | Overlapping confidence intervals confirm no significant difference |
| **Power Analysis** | Experiment is underpowered — larger sample needed |
| **Regression** | `add_to_cart` and `searches` are the strongest predictors of purchases |
| **Classification** | Models can distinguish Control vs Test from metrics, suggesting funnel behavior differs |
| **Funnel Analysis** | Both campaigns have similar end-to-end conversion, but differ in intermediate stages |

### Decision Framework

Since there is **no statistically significant difference** in the primary KPI (purchases), the business should:

1. **Not rush to declare a winner** — the sample size is too small
2. **Evaluate secondary KPIs** — look at CTR, CPA, and funnel stage conversion
3. **Extend the experiment** — run for at least 60 days per group
4. **Consider segment-level analysis** — different customer segments may respond differently

### Overall Verdict

> **Neither campaign shows a clear statistical advantage in purchase volume with the current data. More data is needed before making a conclusive business decision. In the interim, the campaign with lower CPA and higher CTR is preferred from a cost-efficiency perspective.**
